# Fraud Feature Engineering

## Objective
Create fraud intelligence features that simulate real analyst signals: combined risk, velocity spikes, geography anomalies, night-device risk, and amount-to-velocity behaviour.


In [ ]:
# Import the core libraries used for data analysis and visualisation.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make notebook tables easier to inspect during review.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Add the project src folder so notebook code reuses production pipeline logic.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT / 'src'))

from fraud_pipeline import engineer_features, build_feature_list


In [ ]:
# Load all raw datasets from the project data folder.
# The fallback keeps the notebook working if the older data/raw/raw layout exists.
def load_csv(name):
    clean_path = REPO_ROOT / 'data' / 'raw' / name
    nested_path = REPO_ROOT / 'data' / 'raw' / 'raw' / name
    path = clean_path if clean_path.exists() else nested_path
    return pd.read_csv(path)

customers = load_csv('customers.csv')
merchants = load_csv('merchants.csv')
transactions = load_csv('transactions.csv')

print('Customers shape:', customers.shape)
print('Merchants shape:', merchants.shape)
print('Transactions shape:', transactions.shape)


## Feature Engineering Approach
The raw data is clean, so this stage focuses on creating meaningful fraud-risk signals rather than fixing data quality problems.


In [ ]:
# Create engineered features using the same function used by the dashboard.
features_df = engineer_features(transactions)

# Show the new fields added for modelling.
engineered_columns = [col for col in features_df.columns if col not in transactions.columns]
display(features_df[engineered_columns].head())


In [ ]:
# Review the complete feature list passed into feature selection.
model_features = build_feature_list()
print('Number of candidate model features:', len(model_features))
print(model_features)


In [ ]:
# Validate engineered feature behaviour by comparing fraud vs legitimate averages.
comparison = features_df.groupby('fraud_label')[engineered_columns].mean().T
comparison.columns = ['Legitimate Mean', 'Fraud Mean']
display(comparison.sort_values('Fraud Mean', ascending=False))


## Feature Engineering Insight
These features make the model more aligned with real fraud operations because they represent analyst concepts: risk concentration, abnormal location, transaction bursts, and unusual activity timing.
